In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

class FlightInfo(BaseModel):
    flight_time: int = Field(description="평균 소요 시간")

async def get_flight_time(x):
    parser = PydanticOutputParser(pydantic_object=FlightInfo)
    
    template1 = """
    한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 알려줘.
    반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
    {format_instructions}
    """
    
    prompt1 = PromptTemplate(
        template=template1,
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain1 = prompt1 | llm | parser

    resp = await chain1.ainvoke({"country": x["country"]})
    return resp.flight_time
    
template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt2 = PromptTemplate.from_template(template2)

combined_chain = (
    RunnablePassthrough.assign(
        flight_time=get_flight_time
    )
    | prompt2 
    | llm 
    | StrOutputParser()
)

result = combined_chain.ainvoke({"country": "스페인"})
print(await result)